In [8]:
import pandas as pd


In [9]:
file_path = r"C:\Users\amand\OneDrive\Documents\Electricity Bill and Consumption Analyzer\processed data\cleaned_household_power.csv"

# Peek at the column names
temp_df = pd.read_csv(file_path, nrows=5)
print("Columns in your CSV file:")
print(temp_df.columns)

Columns in your CSV file:
Index(['Unnamed: 0', 'Date', 'Time', 'Global_active_power',
       'Global_reactive_power', 'Voltage', 'Global_intensity',
       'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'],
      dtype='object')


In [10]:

df = pd.read_csv(file_path)

# Drop the old 'Unnamed: 0' index column if it's there
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# 2. Combine Date and Time into a proper datetime column
df['datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'], 
    format='%d/%m/%Y %H:%M:%S', 
    errors='coerce'
)

# Set datetime as the index
df = df.set_index('datetime')

#Drop the old standalone Date and Time columns
df = df.drop(columns=['Date', 'Time'])

# Verify the index type and preview
print("Index type:", type(df.index))
print(df.head())

Index type: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>
                     Global_active_power  Global_reactive_power  Voltage  \
datetime                                                                   
2006-12-16 17:24:00                4.216                  0.418   234.84   
2006-12-16 17:25:00                5.360                  0.436   233.63   
2006-12-16 17:26:00                5.374                  0.498   233.29   
2006-12-16 17:27:00                5.388                  0.502   233.74   
2006-12-16 17:28:00                3.666                  0.528   235.68   

                     Global_intensity  Sub_metering_1  Sub_metering_2  \
datetime                                                                
2006-12-16 17:24:00              18.4             0.0             1.0   
2006-12-16 17:25:00              23.0             0.0             1.0   
2006-12-16 17:26:00              23.0             0.0             2.0   
2006-12-16 17:27:00              23.

In [12]:
df['Energy_kWh'] = df['Global_active_power'] / 60

In [13]:
daily_df = df['Energy_kWh'].resample('D').agg(
    daily_consumption_kwh='sum',
    valid_readings_count='count'
)

In [14]:
EXPECTED_MINUTES = 1440
daily_df['is_complete'] = daily_df['valid_readings_count'] == EXPECTED_MINUTES

In [15]:
print(f"Total days analyzed: {len(daily_df)}")
print(f"Incomplete days flagged: (~{(~daily_df['is_complete']).sum()} days)")

Total days analyzed: 1442
Incomplete days flagged: (~2 days)


In [16]:
daily_df.reset_index().to_csv('daily_household_power_consumption.csv', index=False)
print("Daily table successfully saved as 'daily_household_power_consumption.csv'!")

Daily table successfully saved as 'daily_household_power_consumption.csv'!


In [19]:
daily_df = pd.read_csv('C:\\Users\\amand\\OneDrive\\Documents\\Electricity Bill and Consumption Analyzer\\processed data\\daily_household_power_consumption.csv', parse_dates=['datetime'], index_col='datetime')

# Verify it loaded correctly
print(daily_df.head())

            daily_consumption_kwh  valid_readings_count  is_complete
datetime                                                            
2006-12-16              20.152933                   396        False
2006-12-17              56.507667                  1440         True
2006-12-18              36.730433                  1440         True
2006-12-19              27.769900                  1440         True
2006-12-20              37.095800                  1440         True


In [20]:
TARIFF_PER_KWH = 0.50 #to set the tariff per kWh

In [ ]:
daily_df['daily_cost'] = daily_df['daily_consumption_kwh'] * TARIFF_PER_KWH #calculate the daily cost based on the daily consumption and the tariff per kWh

In [22]:
monthly_df = daily_df.resample('ME').agg(
    monthly_consumption_kwh=('daily_consumption_kwh', 'sum'),
    monthly_cost=('daily_cost', 'sum'),
    recorded_days=('daily_consumption_kwh', 'count'),
    complete_days=('is_complete', 'sum')
)

I used resampling to group my daily records into standard calendar months because utility companies bill on a monthly cycle, and my goal is to predict next month's bill. This automatically bundles all the days of each month together, allowing Pandas to easily sum up total monthly consumption and costs while handling different month lengths without complex custom code.

In [ ]:
monthly_df['days_in_month'] = monthly_df.index.days_in_month
monthly_df['is_partial_month'] = monthly_df['recorded_days'] < monthly_df['days_in_month'] #Identified partial months by comparing recorded days against actual days in month

In [24]:
monthly_df['monthly_cost'] = monthly_df['monthly_cost'].round(2)
monthly_df['monthly_consumption_kwh'] = monthly_df['monthly_consumption_kwh'].round(2) #Rounded monetary and consumption values for clean display

In [25]:
print(monthly_df[['monthly_consumption_kwh', 'monthly_cost', 'is_partial_month']].head(12))

            monthly_consumption_kwh  monthly_cost  is_partial_month
datetime                                                           
2006-12-31                   696.98        348.49              True
2007-01-31                  1150.22        575.11             False
2007-02-28                   941.52        470.76             False
2007-03-31                   981.05        490.52             False
2007-04-30                   640.03        320.02             False
2007-05-31                   733.48        366.74             False
2007-06-30                   595.44        297.72             False
2007-07-31                   496.80        248.40             False
2007-08-31                   568.78        284.39             False
2007-09-30                   697.93        348.97             False
2007-10-31                   821.33        410.66             False
2007-11-30                   932.01        466.01             False


In [26]:
monthly_df.reset_index().to_csv('monthly_energy_costs.csv', index=False)
print("\nMonthly energy cost table successfully saved as 'monthly_energy_costs.csv'!")


Monthly energy cost table successfully saved as 'monthly_energy_costs.csv'!
